In [1]:
from openai import OpenAI
import os 
from dotenv import load_dotenv
from minsearch import Index
from pathlib import Path
load_dotenv(Path('..') / '.env')

client = OpenAI()

# Pulling data from Github

Pulling from a specific commit to make sure readme is the same  

In [2]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

##### Parse the files and create documents
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

In [3]:
# Q1. How many lesson pages
len(documents)

72

In [4]:
index = Index(
    text_fields=['content'],
    keyword_fields=['filename']
)
index.fit(documents)

In [5]:
# Q2 - What's the filename of the first result?

question = "How does the agentic loop keep calling the model until it stops?"

search_results = index.search(
    question,
    num_results=1
)

search_results

[{'content': '# The Agentic Loop\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=ePlQUcTPPjw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we did function calling by hand. We sent a\nmessage and got back a function call. We ran it, sent the result back,\nand got the answer.\n\nThat works for one function call. It breaks down when the model wants\nto search several times, or when the first search misses the answer.\nWe don\'t know in advance how many calls the model will want. So we\nneed a loop that keeps calling the model and running tools until it\'s\ndone. An agent is exactly that.\n\n## Anatomy of an agent\n\nWith the LLM in the driver\'s seat, we have an agent. It\'s an AI\nassistant whose goal is to help the user.\n\nAn agent has three parts:\n\n- Instructions, the role and behavior we want. We pass this as the\n  `developer` message. The better the instructions, the better the\n  agent helps.\n- Tools, the functions the agent can call to carry

In [6]:
# Q3 Use gpt-5.4-mini. How many input (prompt) tokens did we send to the model 
# for this request?

# Getting RAG script from Github
! wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py

--2026-06-22 13:16:22--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.108.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2134 (2.1K) [text/plain]
Saving to: ‘rag_helper.py’

rag_helper.py       100%[===================>]   2.08K  --.-KB/s    in 0s      

2026-06-22 13:16:22 (50.5 MB/s) - ‘rag_helper.py’ saved [2134/2134]



In [ ]:
from rag_helper_edited_q3 import RAGBase

rag = RAGBase(
    index=index,
    model="gpt-5.4-mini",
    llm_client=client)

answer, input_tokens, output_tokens, total_tokens = rag.rag("How does the agentic loop keep calling the model until it stops?")
print(f"Answer: {answer}")
print(f"Input tokens: {input_tokens}")
print(f"Output tokens: {output_tokens}")
print(f"Total tokens: {total_tokens}")

Answer: It keeps calling the model in a `while True` loop. After each response, it checks whether the model produced any `function_call` items. If it did, the code runs the tool, appends the tool output to `messages`, and loops again.

It stops when a turn returns **no function calls**:

- `has_function_calls = False`
- if no `function_call` appears, `break`

So the model decides when it’s done, and the loop exits when the response is just a final message.
Input tokens: 7136
Output tokens: 111
Total tokens: 7247


In [ ]:
# Q4 How many chunks do you get?

from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)
print(f"Number of chunks: {len(chunks)}")

# index chunks
index2 = Index(
    text_fields=['content'],
    keyword_fields=['filename']
)
index2.fit(chunks)

# check llm input token usage
rag2 = RAGBase(
    index=index2,
    model="gpt-5.4-mini",
    llm_client=client)

# Q5 - Compare the input tokens with Q3. How many fewer input tokens does the chunked version send?
answer, input_tokens, output_tokens, total_tokens = rag2.rag("How does the agentic loop keep calling the model until it stops?")
print(f"Answer: {answer}")
print(f"Input tokens: {input_tokens}")
print(f"Output tokens: {output_tokens}")
print(f"Total tokens: {total_tokens}")

Number of chunks: 295
Answer: It keeps a `while True` loop around the model call. After each turn, it checks whether any `function_call` items were returned:

- if there are function calls, it runs them, appends the results to `messages`, and loops again
- if there are no function calls, it breaks out of the loop

So the stop condition is: **no function calls this turn means the model is done**.
Input tokens: 2319
Output tokens: 90
Total tokens: 2409


In [16]:
# Q6 - How many times did the agent call search?

from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
            query,
            num_results=num_results,
        )

agent_tools = Tools()
agent_tools.add_tool(search)

chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)


In [17]:
result = runner.loop(
    prompt="How do I run Olama locally?",
    callback=callback,
)

-> Response received


-> Response received


-> Response received
